# Sinh dữ liệu Mixed Human–AI (Qwen2.5-7B & Llama-3.1-8B) trên Kaggle T4

**Hướng dẫn sử dụng (chỉ cần 3 bước):**
1. Ở cột bên phải mục **Input**, bấm **Upload -> New Dataset**, kéo file `prompts_to_generate.jsonl` (6 MB) vào, đặt tên bất kỳ (ví dụ `viet-ai-prompts`) rồi bấm **Create**.
2. Ở mục **Session options** bên phải: bật **Accelerator: `GPU T4 x2`** và bật **Internet: `On`**.
3. Bấm **Run All** (hoặc khuyên dùng: bấm nút **Save Version -> Save & Run All (Commit)** ở góc trên bên phải để Kaggle tự chạy ngầm trên đám mây kể cả khi bạn tắt máy; chạy xong vào tab **Output** tải file `generated_continuations.jsonl` về!).

In [ ]:
# Cell 1: Cài đặt vLLM tương thích Python 3.13 trên Kaggle
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm", "bitsandbytes"])
print("Đã cài đặt xong thư viện!")


In [ ]:
# Cell 2: Ghi file script kaggle_vllm_generate.py (v3+ chống dừng rỗng bằng logit_bias + retry trên GPU)
from pathlib import Path

SCRIPT_CODE = r"""from __future__ import annotations

import argparse
import gc
import glob
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

DEFAULT_MODEL_MAP = {
    "Qwen/Qwen2.5-7B": "Qwen/Qwen2.5-7B",
    "meta-llama/Llama-3.1-8B": "unsloth/Meta-Llama-3.1-8B",
}

FORBIDDEN_GEN_RE = re.compile(
    r"(?:Dưới đây là|Phần tiếp theo|Tin liên quan|Xem thêm|Tags|Bài viết liên quan)",
    re.IGNORECASE,
)


def find_input_file(explicit_path: str | None) -> Path:
    if explicit_path and Path(explicit_path).exists():
        return Path(explicit_path)
    for fname in ("prompts_delta_to_generate.jsonl", "prompts_to_generate.jsonl"):
        matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
        if matches:
            print(f"[Auto-Detect] Đã tìm thấy file đầu vào tại: {matches[0]}", flush=True)
            return Path(matches[0])
    raise FileNotFoundError(
        "Không tìm thấy prompts_delta_to_generate.jsonl hoặc prompts_to_generate.jsonl trong /kaggle/input/! "
        "Hãy bấm Upload -> New Dataset ở thanh bên phải và tải file prompts lên."
    )


def load_done_ids(output_path: Path) -> Set[str]:
    done: Set[str] = set()
    if output_path.exists():
        with open(output_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    done.add(json.loads(line)["sample_id"])
                except Exception:
                    continue
    return done


def _compute_min_tokens(item: Dict[str, Any], max_tok: int) -> int:
    if item.get("drop_prefix_after_gen", False):
        raw_min = min(260, max(180, int(item.get("target_new_words", 220) * 1.0)))
    else:
        raw_min = min(100, max(55, int(item.get("target_new_words", 100) * 0.8)))
    return max(16, min(max_tok - 16, raw_min))


def _is_acceptable_raw(gen_text: str, is_pure_ai: bool) -> bool:
    if not gen_text:
        return False
    txt = gen_text.replace("\r\n", "\n").replace("\r", "\n").lstrip()
    if not txt or FORBIDDEN_GEN_RE.search(txt):
        return False
    dbl = txt.find("\n\n")
    first_block = txt[:dbl] if dbl != -1 else txt
    min_words = 175 if is_pure_ai else 45
    return len(first_block.split()) >= min_words


def _make_sampling_params(
    SamplingParams: Any,
    item: Dict[str, Any],
    temperature: float,
    top_p: float,
    special_bias: Optional[Dict[int, float]],
    seed: Optional[int] = None,
) -> Any:
    max_tok = int(item.get("max_new_tokens", 380))
    min_tok = _compute_min_tokens(item, max_tok)
    base_kwargs: Dict[str, Any] = {
        "temperature": temperature,
        "top_p": top_p,
        "repetition_penalty": 1.08,
        "ignore_eos": True,
        "min_tokens": min_tok,
        "max_tokens": max_tok,
    }
    if seed is not None:
        base_kwargs["seed"] = seed
    if special_bias:
        try:
            return SamplingParams(**base_kwargs, logit_bias=special_bias)
        except Exception:
            pass
    return SamplingParams(**base_kwargs)


def run_with_vllm(
    hf_repo: str,
    items: List[Dict[str, Any]],
    output_path: Path,
    batch_size: int,
    temperature: float,
    top_p: float,
    allow_4bit: bool = False,
    max_retries: int = 4,
) -> None:
    import torch
    import vllm
    from vllm import LLM, SamplingParams

    print(f"[Env] torch={torch.__version__}, vllm={vllm.__version__}", flush=True)
    num_gpus = torch.cuda.device_count()
    if num_gpus == 0:
        raise RuntimeError("Chưa bật GPU! Hãy chọn Accelerator -> GPU T4 x2 ở thanh bên phải.")

    llm_kwargs: Dict[str, Any] = {
        "model": hf_repo,
        "dtype": "float16",
        "max_model_len": 2048,
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
    }
    if num_gpus >= 2:
        llm_kwargs["tensor_parallel_size"] = 2
        dtype_used = "fp16"
        print(f"[vLLM] Chạy {hf_repo} ở fp16 trên 2 GPU (TP=2)...", flush=True)
    elif allow_4bit:
        llm_kwargs["quantization"] = "bitsandbytes"
        llm_kwargs["load_format"] = "bitsandbytes"
        dtype_used = "4bit_bnb"
        print(f"[vLLM] Chỉ có 1 GPU (--allow-4bit) -> Chạy {hf_repo} ở 4-bit bitsandbytes...", flush=True)
    else:
        raise RuntimeError("Cần 2 GPU T4 (hoặc truyền --allow-4bit). Không tự rơi xuống 4-bit ngầm.")

    llm = LLM(**llm_kwargs)
    tok = llm.get_tokenizer()
    special_ids = getattr(tok, "all_special_ids", []) or []
    special_bias: Optional[Dict[int, float]] = {int(tid): -100.0 for tid in special_ids} if special_ids else None
    print(f"[vLLM] Chặn {len(special_ids)} special tokens bằng logit_bias + ignore_eos=True.", flush=True)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    for start in range(0, len(items), batch_size):
        chunk = items[start : start + batch_size]
        prompts = [x["prefix_text"].rstrip() for x in chunk]
        params_list = [_make_sampling_params(SamplingParams, x, temperature, top_p, special_bias, seed=42) for x in chunk]
        try:
            outputs = llm.generate(prompts, params_list)
        except Exception:
            special_bias = None
            params_list = [_make_sampling_params(SamplingParams, x, temperature, top_p, None, seed=42) for x in chunk]
            outputs = llm.generate(prompts, params_list)

        best_texts = [out.outputs[0].text if out.outputs else "" for out in outputs]

        for attempt in range(1, max_retries + 1):
            bad_indices = [
                idx
                for idx, (it, txt) in enumerate(zip(chunk, best_texts))
                if not _is_acceptable_raw(txt, bool(it.get("drop_prefix_after_gen", False)))
            ]
            if not bad_indices:
                break
            retry_temp = min(0.95, round(temperature + 0.05 * attempt, 2))
            print(
                f"[{hf_repo}] Retry lần {attempt}/{max_retries} cho {len(bad_indices)}/{len(chunk)} mẫu ngắn/rỗng (temp={retry_temp})...",
                flush=True,
            )
            sub_prompts = [prompts[i] for i in bad_indices]
            sub_params = [
                _make_sampling_params(SamplingParams, chunk[i], retry_temp, top_p, special_bias, seed=42 + attempt * 97 + i)
                for i in bad_indices
            ]
            sub_outs = llm.generate(sub_prompts, sub_params)
            for i, out in zip(bad_indices, sub_outs):
                cand = out.outputs[0].text if out.outputs else ""
                if len(cand.split()) > len(best_texts[i].split()):
                    best_texts[i] = cand

        with open(output_path, "a", encoding="utf-8") as out_f:
            for item, gen_text in zip(chunk, best_texts):
                rec = {
                    **item,
                    "hf_repo_used": hf_repo,
                    "gen_config": {
                        "repo": hf_repo,
                        "vllm_version": vllm.__version__,
                        "dtype": dtype_used,
                        "temperature": temperature,
                        "top_p": top_p,
                        "repetition_penalty": 1.08,
                        "ignore_eos": True,
                    },
                    "raw_continuation": gen_text,
                }
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        ok_cnt = sum(
            1 for it, txt in zip(chunk, best_texts) if _is_acceptable_raw(txt, bool(it.get("drop_prefix_after_gen", False)))
        )
        print(
            f"[{hf_repo}] Đã lưu {min(start + batch_size, len(items))}/{len(items)} mẫu (đạt độ dài ngay trên GPU: {ok_cnt}/{len(chunk)}).",
            flush=True,
        )

    del llm
    gc.collect()
    torch.cuda.empty_cache()


def run_with_transformers_fallback(
    hf_repo: str,
    items: List[Dict[str, Any]],
    output_path: Path,
    temperature: float,
    top_p: float,
) -> None:
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print(f"[Transformers Fallback] Đang nạp {hf_repo} với device_map=auto (fp16)...", flush=True)
    tok = AutoTokenizer.from_pretrained(hf_repo, padding_side="left", trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    suppress = list(getattr(tok, "all_special_ids", []) or [])
    model = AutoModelForCausalLM.from_pretrained(
        hf_repo,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    bs = 8
    for start in range(0, len(items), bs):
        chunk = items[start : start + bs]
        prompts = [x["prefix_text"].rstrip() for x in chunk]
        max_new = max(int(x.get("max_new_tokens", 380)) for x in chunk)
        min_new = min(_compute_min_tokens(x, max_new) for x in chunk)
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)
        in_len = enc.input_ids.shape[1]
        gen_kw = dict(
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.08,
            min_new_tokens=min_new,
            max_new_tokens=max_new,
            pad_token_id=tok.pad_token_id,
        )
        if suppress:
            gen_kw["suppress_tokens"] = suppress
        with torch.no_grad():
            gen_ids = model.generate(**enc, **gen_kw)
        decoded = tok.batch_decode(gen_ids[:, in_len:], skip_special_tokens=True)
        with open(output_path, "a", encoding="utf-8") as out_f:
            for item, gen_text in zip(chunk, decoded):
                rec = {
                    **item,
                    "hf_repo_used": hf_repo,
                    "gen_config": {
                        "repo": hf_repo,
                        "transformers_version": transformers.__version__,
                        "dtype": "fp16_hf_fallback",
                        "temperature": temperature,
                        "top_p": top_p,
                        "repetition_penalty": 1.08,
                    },
                    "raw_continuation": gen_text,
                }
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        if (start // bs) % 5 == 0 or start + bs >= len(items):
            print(f"[{hf_repo}] Đã lưu {min(start + bs, len(items))}/{len(items)} mẫu.", flush=True)

    del model, tok
    gc.collect()
    torch.cuda.empty_cache()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", default=None, help="Tự động tìm trong /kaggle/input nếu bỏ trống")
    parser.add_argument("--output", default="/kaggle/working/generated_continuations.jsonl", type=Path)
    parser.add_argument("--batch-size", type=int, default=200)
    parser.add_argument("--temperature", type=float, default=0.7)
    parser.add_argument("--top-p", type=float, default=0.9)
    parser.add_argument("--allow-4bit", action="store_true")
    args = parser.parse_args()

    in_path = find_input_file(args.input)
    done_ids = load_done_ids(args.output)
    print(f"Số mẫu đã hoàn thành từ trước: {len(done_ids)}", flush=True)

    grouped: Dict[str, List[Dict[str, Any]]] = {}
    with open(in_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj["sample_id"] not in done_ids:
                grouped.setdefault(obj["generator"], []).append(obj)

    for gen_key, pending_items in grouped.items():
        hf_repo = DEFAULT_MODEL_MAP.get(gen_key, gen_key)
        print(f"=== Bắt đầu sinh {len(pending_items)} mẫu cho {gen_key} ({hf_repo}) ===", flush=True)
        try:
            run_with_vllm(hf_repo, pending_items, args.output, args.batch_size, args.temperature, args.top_p, args.allow_4bit)
        except Exception as exc:
            print(f"[Cảnh báo] vLLM gặp lỗi ({exc}) -> Chuyển sang Transformers...", flush=True)
            done_now = load_done_ids(args.output)
            rem = [x for x in pending_items if x["sample_id"] not in done_now]
            if rem:
                run_with_transformers_fallback(hf_repo, rem, args.output, args.temperature, args.top_p)

    print("HOÀN TẤT 100%! Hãy tải file /kaggle/working/generated_continuations.jsonl về máy.", flush=True)


if __name__ == "__main__":
    main()
"""

Path("kaggle_vllm_generate.py").write_text(SCRIPT_CODE, encoding="utf-8")
print("Đã tạo xong kaggle_vllm_generate.py (v3+)!")


In [ ]:
# Cell 3: Chạy tiến trình sinh toàn bộ dữ liệu v3
import subprocess, sys
subprocess.run([sys.executable, "kaggle_vllm_generate.py"], check=True)


In [ ]:
# Cell 4: Kiểm tra kết quả đầu ra
from pathlib import Path
out = Path("/kaggle/working/generated_continuations.jsonl")
lines = out.read_text(encoding="utf-8").splitlines()
print(f"Tổng số dòng đã sinh: {len(lines)}")
print("Mẫu đầu tiên:", lines[0][:300] if lines else "Rỗng")
